In [1]:
import os
import base64
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow  # pyright: ignore[reportMissingImports]
from googleapiclient.discovery import build

SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']

In [2]:
def gmail_authenticate():
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    return build('gmail', 'v1', credentials=creds)

In [3]:
def get_recent_emails(service, max_results=4):
    results = service.users().messages().list(
        userId='me', maxResults=max_results, labelIds=['INBOX']
    ).execute()
    messages = results.get('messages', [])

    emails = []
    for msg in messages:
        msg_data = service.users().messages().get(
            userId='me', id=msg['id'], format='full'
        ).execute()

        headers = msg_data['payload']['headers']
        subject = next((h['value'] for h in headers if h['name'] == 'Subject'), 'بدون عنوان')
        sender = next((h['value'] for h in headers if h['name'] == 'From'), 'غير معروف')

        body = extract_body(msg_data['payload'])
        emails.append({'subject': subject, 'from': sender, 'body': body})
    return emails

In [4]:
def extract_body(payload):
    if 'parts' in payload:
        for part in payload['parts']:
            if part['mimeType'] == 'text/plain':
                data = part['body'].get('data', '')
                if data:
                    return base64.urlsafe_b64decode(data).decode('utf-8', errors='ignore')
    else:
        data = payload['body'].get('data', '')
        if data:
            return base64.urlsafe_b64decode(data).decode('utf-8', errors='ignore')
    return ''

In [5]:
import ollama

def analyze_email(email):
    prompt = f"""حلل الإيميل ده وارجعلي:
1. ملخص قصير (سطرين كحد أقصى)
2. التصنيف: (عمل / شخصي / إعلانات / عاجل / سبام)
3. مستوى الأهمية: (عالي / متوسط / منخفض)

من: {email['from']}
الموضوع: {email['subject']}
المحتوى: {email['body'][:1500]}
"""
    response = ollama.chat(
        model='gemma4:e2b',
        messages=[{'role': 'user', 'content': prompt}]
    )
    return response['message']['content']

def main():
    service = gmail_authenticate()
    emails = get_recent_emails(service, max_results=4)

    results = []
    for email in emails:
        analysis = analyze_email(email)
        results.append({'email': email, 'analysis': analysis})

    for r in results:
        print(f"\n{'='*50}")
        print(f"من: {r['email']['from']}")
        print(f"الموضوع: {r['email']['subject']}")
        print(f"\nالتحليل:\n{r['analysis']}")

if __name__ == '__main__':
    main()


من: Ollama <hello@ollama.com>
الموضوع: GLM 5.3 & 5.3 Flash models are now available

التحليل:
إليك تحليل الإيميل المطلوب:

**1. ملخص قصير (سطرين كحد أقصى)**
أعلنت Ollama عن توفر نماذج الذكاء الاصطناعي الجديدة GLM 5.3 و GLM 5.3 Flash على سحابتها، وهي مُحسّنة للمهام المتعلقة بالبرمجة والعمل كوكلاء (agents). كما يقدم الإيميل إرشادات حول كيفية استخدام هذه النماذج مع أدوات أخرى مثل Claude و Hermes.

**2. التصنيف: (عمل / شخصي / إعلانات / عاجل / سبام)**
عمل

**3. مستوى الأهمية: (عالي / متوسط / منخفض)**
عالي (للمستخدمين المهتمين بالذكاء الاصطناعي والبرمجة)

من: GitHub <noreply@github.com>
الموضوع: [GitHub] A first-party GitHub OAuth application has been added to your account

التحليل:
إليك تحليل الإيميل المطلوب:

**1. ملخص قصير (سطرين كحد أقصى):**
تم إخطارك من GitHub بأنه تم منح تطبيق OAuth (Git Credential Manager) إذنًا للوصول إلى حسابك، مع توفير روابط لمزيد من المعلومات حول الإجراءات الأمنية والوصول.

**2. التصنيف:**
عمل

**3. مستوى الأهمية:**
متوسط (لأنه يتعلق بأمان الحساب، ولكن يجب مراجعة